In [ ]:
from utils.data import ProteinDataset, QueryHomologyDataset
import torch as pt
import numpy as np



data = pt.load(f'./data/pbond0_hbond0.pt', weights_only=False)
datalib = ProteinDataset(data)
lib_map = np.arange(len(datalib), dtype=np.int64)
print('number of proteins:', len(lib_map))
pdb2idx = [(data[2][i], i) for i in range(len(data[2]))] # pdb name -> idx
pdb2idx = dict(pdb2idx)
pair_dataset = QueryHomologyDataset(datalib, './data/tmalign.out', pdb2idx)

number of proteins: 106973


In [2]:
import torch.nn as nn
import torch_geometric.nn as gnn
from torch.nn import TransformerEncoder, TransformerEncoderLayer


class ProteinGCN(nn.Module):
    def __init__(self, embed_dim:int=512, hidden_channels:int=256, out_channels:int=128, num_layers:int=3, num_edge_features:int=10,):
        super().__init__()
        self.emb = nn.Embedding(num_embeddings=21, embedding_dim=embed_dim, padding_idx=0,)
        # node_attr占一维
        self.gcn = gnn.GCN(in_channels=embed_dim+num_edge_features, hidden_channels=hidden_channels, 
                           num_layers=num_layers, out_channels=out_channels,)
        self.shared = nn.Sequential(nn.Linear(4*out_channels, out_channels), nn.SiLU(),)
        self.tm_head = nn.Linear(out_channels, 1,)
        self.seq_head = nn.Linear(out_channels, 1,)
        # encoder_layer = TransformerEncoderLayer(d_model=embed_dim, nhead=8, dim_feedforward=embed_dim*4, 
                                                # activation='gelu', batch_first=True,)
        # self.encoder = TransformerEncoder(encoder_layer, num_layers=1)
        # self.mlp = nn.Linear(embed_dim, 21)

    def embed(self, seq_mask, mode:str='train'):
        seq, mask = seq_mask
        embedding = self.emb(seq) # [batch_size, seq_len, emb_dim]
        embedding = embedding * mask.unsqueeze(-1) # mask: [batch_size, seq_len, 1]
        if mode == 'emb':
            embedding = embedding.sum(dim=1) # [batch_size, emb_dim]
        return embedding

    def encode_protein(self, seq, mask, graph):
        x, edge_idx, edge_attr, batch, node2seq = graph.x, graph.edge_index, graph.edge_attr, graph.batch, graph.node2seq
        emb = self.embed((seq, mask))
        B, L, D = emb.shape
        emb_flat = emb.view(-1, D)
        flat_idx = batch * L + node2seq
        node_emb = emb_flat[flat_idx]
        x = pt.cat([node_emb, x], dim=-1)
        x = self.gcn(x, edge_idx, edge_attr=edge_attr, batch=batch)
        x = gnn.global_mean_pool(x, batch)
        return x        

    def forward(self, data, mode:str='finetuning'):
        if mode == 'pretraining':
            seqs_pad, masks_pad = data
            embs = self.emb(seqs_pad) # [batch_size, seq_len, emb_dim]
            embs = self.encoder(embs, src_key_padding_mask=~masks_pad) # [batch_size, seq_len, emb_dim]
            outputs = self.mlp(embs) # [batch_size, seq_len, 21]
            return outputs
        elif mode == 'finetuning':            
            (seqs, masks, graphs), (inv_i, inv_j) = data
            prot_repr = self.encode_protein(seqs, masks, graphs)
            x_i = prot_repr[inv_i]
            x_j = prot_repr[inv_j]
            feature = pt.cat([x_i, x_j, x_i-x_j, x_i*x_j], dim=-1)
            shared = self.shared(feature)
            tm_score = self.tm_head(shared).squeeze(-1)
            seq_score = self.seq_head(shared).squeeze(-1)
            return tm_score, seq_score
        else:
            raise ValueError(f'Unknown mode: {mode}')

In [3]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from utils.data import pair_collate_fun, collate_fun_emb




libloader = DataLoader(datalib, batch_size=512, shuffle=False, collate_fn=collate_fun_emb, num_workers=6)
pair_map = np.arange(len(pair_dataset), dtype=np.int64)
print('len pair_dataset:', len(pair_map))
_, test_map = train_test_split(lib_map, test_size=1024, random_state=42)
train_set = ProteinPairDataset(pair_dataset, mapping=pair_map)
test_set = ProteinDataset(datalib, mapping=test_map)
train_loader = DataLoader(train_set, batch_size=384, shuffle=True, 
                          collate_fn=pair_collate_fun(datalib), drop_last=True, num_workers=6)
test_loader = DataLoader(test_set, batch_size=512, shuffle=False, 
                         collate_fn=collate_fun_emb, num_workers=6)

len pair_dataset: 1214640


In [4]:
from tqdm import tqdm
from utils.tools import gen_embeddings, build_idx, calculate_remote_homology_score, save_model
from dataclasses import dataclass
import torch as pt
import torch.nn as nn
from torch.utils.data import DataLoader


@dataclass
class TrainConfig:
    epochs: int = 10
    gpu: int = 6
    model_name: str = "gcn"
    lr: float = 1e-3
    pdb_root: str = "../../data/pdb"
    tmalign_path: str = "./TMalign"
    topk: int = 12
    tmalign_reference: int = 1
    tmalign_workers: int = None  # None 表示自动选择


class RegressionTrainer:
    def __init__(self, model, config: TrainConfig):
        super().__init__()
        self.model = model
        self.config = config
        self.log_file = open(f"{config.model_name}.txt", "w", encoding="utf-8")
        self.max_score = float("-inf")
        self.criterion = nn.SmoothL1Loss()
        self.optimizer = pt.optim.AdamW(self.model.parameters(), lr=config.lr)
        self.scheduler = pt.optim.lr_scheduler.StepLR(
            self.optimizer, step_size=1, gamma=0.1
        )

    def close(self):
        if not self.log_file.closed:
            self.log_file.close()

    def train(
        self,
        train_loader: DataLoader,
        test_loader: DataLoader,
        lib_loader: DataLoader,
        test_set,
        datalib,
    ):
        try:
            for epoch in range(self.config.epochs):
                print(f"======================= Train epoch {epoch + 1} =======================")
                self.log_file.write(f"\nEpoch {epoch + 1} Training\n")
                train_loss = []
                self.model.train()
                for i, batch in enumerate(tqdm(train_loader, unit="batch")):
                    prot, inv, score = batch
                    prot = [x.to(self.config.gpu) for x in prot]
                    inv = [x.to(self.config.gpu) for x in inv]
                    score = score.to(self.config.gpu)
                    output = self.model((prot, inv))
                    tm_score, seq_score = output
                    tm_loss = self.criterion(tm_score, score[:, 0])
                    seq_loss = self.criterion(seq_score, score[:, 1])
                    loss = tm_loss + seq_loss
                    self.optimizer.zero_grad()
                    loss.backward()
                    self.optimizer.step()
                    train_loss.append(loss.item())
                    if (i + 1) % 500 == 0:
                        avg_loss = float(pt.tensor(train_loss).mean())
                        msg = f"Epoch [{epoch + 1}/{self.config.epochs}], Train Loss: {avg_loss:.4f}"
                        print(msg)
                        self.log_file.write(msg + "\n")
                        train_loss = []

                self.scheduler.step()
                print(f"======================= Test epoch {epoch + 1} =======================")
                self.log_file.write(f"\nEpoch {epoch + 1} Testing\n")
                pt.cuda.empty_cache()
                self.model.eval()
                with pt.no_grad():
                    embs_lib = gen_embeddings(self.model, lib_loader, self.config.gpu)
                    embs_test = gen_embeddings(self.model, test_loader, self.config.gpu)
                I, _ = build_idx(embs_lib, embs_test, self.config.gpu)
                score = calculate_remote_homology_score(
                    query=test_set,
                    database=datalib,
                    idx=I,
                    k=self.config.topk,
                    pdb_root=self.config.pdb_root,
                    tmalign_path=self.config.tmalign_path,
                    reference=self.config.tmalign_reference,
                    num_workers=self.config.tmalign_workers,
                )
                msg = f"Remote homologous score: {score:.6f}"
                print(msg)
                self.log_file.write(msg + "\n")
                if score > self.max_score:
                    self.max_score = score
                    save_model(self.model, self.config.model_name, epoch)
                print("======================================================================")
        finally:
            self.close()


if __name__ == "__main__":
    config = TrainConfig()
    pt.cuda.set_device(config.gpu)
    model = ProteinGCN().cuda(config.gpu)
    trainer = RegressionTrainer(model, config)
    trainer.train(
        train_loader=train_loader,
        test_loader=test_loader,
        lib_loader=libloader,
        test_set=test_set,
        datalib=datalib,
    )
    pt.cuda.empty_cache()

======================= Train epoch 1 =======================


 16%|█▌        | 501/3163 [00:50<04:25, 10.04batch/s]

Epoch [1/10], Train Loss: 0.0063


 32%|███▏      | 1001/3163 [01:40<03:35, 10.02batch/s]

Epoch [1/10], Train Loss: 0.0047


...

Epoch [10/10], Train Loss: 0.0037


100%|██████████| 3163/3163 [05:23<00:00,  9.78batch/s]

======================= Test epoch 10 =======================


Searching time:  0:00:00.010184


Remote homologous score: -0.230609
